# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [ ]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 48.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=625cd2e4-5ab5-46a5-aebe-a62748121fb3
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 77.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 17.5MB/s]


In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1)
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (122 s)


{'clean': 78.7, 'linf': 48.4, 'l2': 59.8, 'l1': 22.4, 'union': 22.3}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203 (41 s)
epoch 2: loss 1.181 (81 s)
epoch 3: loss 1.152 (122 s)
epoch 4: loss 1.126 (162 s)
epoch 5: loss 1.097 (203 s)
epoch 6: loss 1.068 (244 s)
epoch 7: loss 1.049 (284 s)
epoch 8: loss 1.030 (325 s)
epoch 9: loss 1.013 (366 s)
epoch 10: loss 1.008 (406 s)
clean 78.5  linf 44.7  l2 57.2  l1 20.4  union 20.3   (122 s)


{'clean': 78.5, 'linf': 44.7, 'l2': 57.2, 'l1': 20.4, 'union': 20.3}

In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.897 (40 s)
epoch 2: loss 0.826 (81 s)
epoch 3: loss 0.787 (121 s)
epoch 4: loss 0.754 (162 s)
epoch 5: loss 0.715 (202 s)
epoch 6: loss 0.675 (243 s)
epoch 7: loss 0.631 (283 s)
epoch 8: loss 0.610 (324 s)
epoch 9: loss 0.591 (364 s)
epoch 10: loss 0.583 (405 s)
clean 82.9  linf 32.3  l2 60.2  l1 27.8  union 24.7   (122 s)


{'clean': 82.9, 'linf': 32.3, 'l2': 60.2, 'l1': 27.8, 'union': 24.7}

In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.233 (41 s)
epoch 2: loss 0.870 (82 s)
epoch 3: loss 0.599 (123 s)
epoch 4: loss 0.449 (165 s)
epoch 5: loss 0.345 (206 s)
epoch 6: loss 0.255 (247 s)
epoch 7: loss 0.206 (288 s)
epoch 8: loss 0.168 (330 s)
epoch 9: loss 0.154 (371 s)
epoch 10: loss 0.152 (412 s)
clean 84.0  linf 0.0  l2 0.0  l1 21.5  union 0.0   (122 s)


{'clean': 84.0, 'linf': 0.0, 'l2': 0.0, 'l1': 21.5, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q, logits_r = model(x_q), model(x_r)
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    loss = torch.max(ce_q, ce_r).mean()
    mask = logits_q.argmax(1) == y
    if mask.any():
        log_p_q = F.log_softmax(logits_q[mask], dim=1)
        log_p_r = F.log_softmax(logits_r[mask], dim=1)
        kl = (log_p_q.exp() * (log_p_q - log_p_r)).sum(1).mean()
        loss = loss + lam * kl
    return loss


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    cos = (g_n * g_a).sum() / (g_n.norm() * g_a.norm() + 1e-12)
    return cos * g_n if cos > 0 else torch.zeros_like(g_n)

In [ ]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.432 (88 s)
epoch 2: loss 1.330 (175 s)
epoch 3: loss 1.303 (262 s)
epoch 4: loss 1.280 (350 s)
epoch 5: loss 1.254 (438 s)
epoch 6: loss 1.227 (525 s)
epoch 7: loss 1.198 (612 s)
epoch 8: loss 1.185 (700 s)
epoch 9: loss 1.171 (788 s)
epoch 10: loss 1.166 (875 s)
clean 79.2  linf 41.0  l2 61.9  l1 44.3  union 38.8   (122 s)


{'clean': 79.2, 'linf': 41.0, 'l2': 61.9, 'l1': 44.3, 'union': 38.8}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [ ]:
teacher = load_base_model().eval()
for p in teacher.parameters():
    p.requires_grad_(False)

norm_acc = {}


def reset_norm_acc():
    norm_acc.update({n: 0.5 for n in NORMS})


reset_norm_acc()

In [ ]:
def my_loss(model, x, y, adaptive=True, mu=1.0, temp=0.2, momentum=0.9):
    # TODO: your method
    if adaptive:
        w = torch.tensor([1 - norm_acc[n] for n in NORMS], device=x.device)
        probs = F.softmax(w / temp, dim=0)
    else:
        probs = torch.full((len(NORMS),), 1 / len(NORMS), device=x.device)
    assign = torch.multinomial(probs, len(y), replacement=True)
    model.eval()
    x_adv = x.clone()
    for k, n in enumerate(NORMS):
        idx = assign == k
        if idx.any():
            x_adv[idx] = attack(model, x[idx], y[idx], n)
    model.train()
    logits = model(x_adv)
    with torch.no_grad():
        correct = logits.argmax(1) == y
        for k, n in enumerate(NORMS):
            idx = assign == k
            if idx.any():
                norm_acc[n] = momentum * norm_acc[n] + (1 - momentum) * correct[idx].float().mean().item()
        t_logits = teacher(x_adv)
        keep = t_logits.argmax(1) == y
    loss = F.cross_entropy(logits, y)
    if mu > 0 and keep.any():
        kl = F.kl_div(F.log_softmax(logits[keep], dim=1), F.softmax(t_logits[keep], dim=1), reduction="batchmean")
        loss = loss + mu * kl
    return loss

In [ ]:
# model_mine = adv_train(load_base_model(), my_loss)
# test_model(model_mine)

In [ ]:
model_ramp_s1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
test_model(model_ramp_s1)

epoch 1: loss 1.434 (88 s)
epoch 2: loss 1.333 (175 s)
epoch 3: loss 1.298 (263 s)
epoch 4: loss 1.274 (350 s)
epoch 5: loss 1.248 (438 s)
epoch 6: loss 1.221 (525 s)
epoch 7: loss 1.201 (613 s)
epoch 8: loss 1.186 (700 s)
epoch 9: loss 1.178 (788 s)
epoch 10: loss 1.168 (875 s)
clean 79.0  linf 42.2  l2 62.6  l1 46.5  union 40.4   (122 s)


{'clean': 79.0, 'linf': 42.2, 'l2': 62.6, 'l1': 46.5, 'union': 40.4}

In [ ]:
reset_norm_acc()
model_mine = adv_train(load_base_model(), my_loss, seed=0)
test_model(model_mine)
print(norm_acc)

epoch 1: loss 1.308 (44 s)
epoch 2: loss 1.228 (88 s)
epoch 3: loss 1.187 (132 s)
epoch 4: loss 1.160 (176 s)
epoch 5: loss 1.122 (220 s)
epoch 6: loss 1.093 (263 s)
epoch 7: loss 1.075 (306 s)
epoch 8: loss 1.057 (350 s)
epoch 9: loss 1.043 (394 s)
epoch 10: loss 1.035 (437 s)
clean 77.9  linf 43.0  l2 62.9  l1 46.7  union 40.0   (122 s)
{'linf': 0.5646754753185033, 'l2': 0.7429358415207674, 'l1': 0.5895015859045057}


In [ ]:
reset_norm_acc()
model_mine_s1 = adv_train(load_base_model(), my_loss, seed=1)
test_model(model_mine_s1)
print(norm_acc)

epoch 1: loss 1.300 (44 s)
epoch 2: loss 1.229 (88 s)
epoch 3: loss 1.190 (132 s)
epoch 4: loss 1.148 (176 s)
epoch 5: loss 1.117 (219 s)
epoch 6: loss 1.085 (263 s)
epoch 7: loss 1.068 (306 s)
epoch 8: loss 1.055 (349 s)
epoch 9: loss 1.043 (393 s)
epoch 10: loss 1.034 (436 s)
clean 79.3  linf 43.1  l2 63.0  l1 47.3  union 40.0   (122 s)
{'linf': 0.5791727085436388, 'l2': 0.718571615546203, 'l1': 0.6291277480123791}


In [ ]:
reset_norm_acc()
model_uniform = adv_train(load_base_model(), my_loss, seed=0, adaptive=False)
test_model(model_uniform)

epoch 1: loss 1.244 (41 s)
epoch 2: loss 1.175 (81 s)
epoch 3: loss 1.146 (122 s)
epoch 4: loss 1.102 (162 s)
epoch 5: loss 1.072 (203 s)
epoch 6: loss 1.052 (244 s)
epoch 7: loss 1.025 (285 s)
epoch 8: loss 1.014 (325 s)
epoch 9: loss 1.001 (366 s)
epoch 10: loss 0.992 (408 s)
clean 78.0  linf 42.8  l2 62.9  l1 46.9  union 40.4   (122 s)


{'clean': 78.0, 'linf': 42.8, 'l2': 62.9, 'l1': 46.9, 'union': 40.4}

In [ ]:
reset_norm_acc()
model_no_kl = adv_train(load_base_model(), my_loss, seed=0, mu=0.0)
test_model(model_no_kl)
print(norm_acc)

epoch 1: loss 1.256 (44 s)
epoch 2: loss 1.174 (87 s)
epoch 3: loss 1.134 (131 s)
epoch 4: loss 1.106 (175 s)
epoch 5: loss 1.070 (218 s)
epoch 6: loss 1.038 (261 s)
epoch 7: loss 1.018 (304 s)
epoch 8: loss 0.990 (348 s)
epoch 9: loss 0.986 (391 s)
epoch 10: loss 0.982 (434 s)
clean 78.9  linf 41.8  l2 61.9  l1 47.3  union 39.9   (122 s)
{'linf': 0.5765375410143925, 'l2': 0.7420658282030514, 'l1': 0.6342741085577279}


In [ ]:
test_model(model_mine, steps=200, restarts=5)

clean 77.9  linf 42.7  l2 62.8  l1 44.6  union 39.1   (1211 s)


{'clean': 77.9, 'linf': 42.7, 'l2': 62.8, 'l1': 44.6, 'union': 39.1}

# Problem 1: Adversarial Training from Scratch

In [ ]:
def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)

In [ ]:
torch.manual_seed(0)
model_std = adv_train(make_model(), standard_loss, epochs=5)
test_model(model_std)

epoch 1: loss 1.877 (5 s)
epoch 2: loss 1.547 (11 s)
epoch 3: loss 1.391 (16 s)
epoch 4: loss 1.233 (21 s)
epoch 5: loss 1.147 (27 s)
clean 57.6  linf 0.3  l2 7.9  l1 1.3  union 0.3   (123 s)


{'clean': 57.6, 'linf': 0.3, 'l2': 7.9, 'l1': 1.3, 'union': 0.3}

In [ ]:
torch.manual_seed(0)
model_adv = adv_train(make_model(), single_norm_loss, epochs=5, norm="linf")
test_model(model_adv)

epoch 1: loss 2.188 (41 s)
epoch 2: loss 2.096 (81 s)
epoch 3: loss 2.049 (122 s)
epoch 4: loss 2.029 (162 s)
epoch 5: loss 2.014 (203 s)
clean 37.0  linf 24.0  l2 29.6  l1 21.4  union 19.9   (121 s)


{'clean': 37.0, 'linf': 24.0, 'l2': 29.6, 'l1': 21.4, 'union': 19.9}

In [ ]:
def linf_accuracy(model, eps, steps, batch_size=250):
    model.eval()
    torch.manual_seed(0)
    alpha = eps if steps == 1 else 2.5 * eps / steps
    correct = 0
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        x_adv = pgd_linf(model, x, y, eps, alpha, steps, rand_init=steps > 1)
        with torch.no_grad():
            correct += (model(x_adv).argmax(1) == y).sum().item()
    return 100.0 * correct / len(y_test)

In [ ]:
for e in [0, 1, 2, 4, 8, 12, 16]:
    row = [f"eps {e}/255"]
    for name, m in [("standard", model_std), ("linf", model_adv)]:
        row.append(f"{name}: pgd {linf_accuracy(m, e / 255, 50):.1f} fgsm {linf_accuracy(m, e / 255, 1):.1f}")
    print("   ".join(row))

eps 0/255   standard: pgd 57.6 fgsm 57.6   linf: pgd 37.0 fgsm 37.0
eps 1/255   standard: pgd 37.0 fgsm 39.1   linf: pgd 35.4 fgsm 35.4
eps 2/255   standard: pgd 22.5 fgsm 27.9   linf: pgd 33.5 fgsm 33.5
eps 4/255   standard: pgd 5.7 fgsm 11.6   linf: pgd 29.7 fgsm 29.9
eps 8/255   standard: pgd 0.3 fgsm 2.9   linf: pgd 24.2 fgsm 24.7
eps 12/255   standard: pgd 0.0 fgsm 1.1   linf: pgd 18.3 fgsm 20.4
eps 16/255   standard: pgd 0.0 fgsm 0.6   linf: pgd 12.2 fgsm 15.0
